<a href="https://colab.research.google.com/github/abuhurera54-creator/Unsupervised_learning/blob/main/Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ONLINE RETAIL II - CUSTOMER CLUSTERING
# K-MEANS + HIERARCHICAL + DBSCAN + GMM
# WITH GRAPHS AND MODEL COMPARISON
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage


# ============================================================
# 2. LOAD DATASET
# ============================================================

# Put online_retail_II.xlsx in the same folder as this Python file

file_path = "Assignment-1_Data.csv"

# Read the CSV file directly, assuming it contains combined data
df = pd.read_csv(file_path, engine='python', on_bad_lines='skip', sep=';')


# ============================================================
# 3. DISPLAY BASIC INFORMATION
# ============================================================

print("=" * 60)
print("ONLINE RETAIL II DATASET")
print("=" * 60)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Information:")
print(df.info())


# ============================================================
# 4. STANDARDIZE COLUMN NAMES
# ============================================================

# UCI Online Retail II uses "Invoice", "Price",
# and "Customer ID".

df.rename(
    columns={
        "BillNo": "InvoiceNo", # Renamed 'Invoice' to 'InvoiceNo'
        "Date": "InvoiceDate", # Renamed 'Date' to 'InvoiceDate'
        "Price": "UnitPrice", # Renamed 'Price' to 'UnitPrice'
        "Customer ID": "CustomerID" # 'Customer ID' remains 'CustomerID' (for consistency)
    },
    inplace=True
)

print("\nColumns after renaming:")
print(df.columns.tolist())


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())


# ============================================================
# 6. CONVERT DATE AND PRICE COLUMNS
# ============================================================

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

# Convert UnitPrice to numeric, handling potential commas as decimal separators
df["UnitPrice"] = (
    df["UnitPrice"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)


# ============================================================
# 7. REMOVE MISSING CUSTOMER IDs
# ============================================================

df = df.dropna(
    subset=["CustomerID"]
)

print("\nShape after removing missing Customer IDs:")
print(df.shape)


# ============================================================
# 8. REMOVE CANCELLED TRANSACTIONS
# ============================================================

df = df[
    ~df["InvoiceNo"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
]

print("\nShape after removing cancelled transactions:")
print(df.shape)


# ============================================================
# 9. REMOVE INVALID QUANTITY AND PRICE
# ============================================================

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
]

print("\nShape after removing invalid transactions:")
print(df.shape)


# ============================================================
# 10. CREATE TOTAL AMOUNT
# ============================================================

df["TotalAmount"] = (
    df["Quantity"] *
    df["UnitPrice"]
)


print("\nTotal Amount created successfully.")

print(df[
    [
        "CustomerID",
        "Quantity",
        "UnitPrice",
        "TotalAmount"
    ]
].head())


# ============================================================
# 11. CREATE REFERENCE DATE
# ============================================================

reference_date = (
    df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

print("\nReference Date:")
print(reference_date)


# ============================================================
# 12. CREATE RFM DATASET
# ============================================================

# R = Recency
# F = Frequency
# M = Monetary

rfm = df.groupby("CustomerID").agg({

    "InvoiceDate": lambda x:
        (reference_date - x.max()).days,

    "InvoiceNo": "nunique",

    "TotalAmount": "sum"
})


# ============================================================
# 13. RENAME RFM COLUMNS
# ============================================================

rfm.rename(
    columns={
        "InvoiceDate": "Recency",
        "InvoiceNo": "Frequency",
        "TotalAmount": "Monetary"
    },
    inplace=True
)


# ============================================================
# 14. DISPLAY RFM
# ============================================================

print("\n" + "=" * 60)
print("RFM DATASET")
print("=" * 60)

print(rfm.head())

print("\nRFM Shape:")
print(rfm.shape)

print("\nRFM Statistics:")
print(rfm.describe())


# ============================================================
# 15. RFM DISTRIBUTION GRAPHS
# ============================================================

plt.figure(figsize=(8, 5))

sns.histplot(
    rfm["Recency"],
    bins=30,
    kde=True
)

plt.title("Recency Distribution")
plt.xlabel("Recency (Days)")
plt.ylabel("Number of Customers")

plt.show()


plt.figure(figsize=(8, 5))

sns.histplot(
    rfm["Frequency"],
    bins=30,
    kde=True
)

plt.title("Frequency Distribution")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Customers")

plt.show()


plt.figure(figsize=(8, 5))

sns.histplot(
    rfm["Monetary"],
    bins=30,
    kde=True
)

plt.title("Monetary Distribution")
plt.xlabel("Total Spending")
plt.ylabel("Number of Customers")

plt.show()


# ============================================================
# 16. REMOVE EXTREME OUTLIERS USING IQR
# ============================================================

for column in [
    "Recency",
    "Frequency",
    "Monetary"
]:

    Q1 = rfm[column].quantile(0.25)

    Q3 = rfm[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR

    upper_limit = Q3 + 1.5 * IQR

    rfm = rfm[
        (rfm[column] >= lower_limit) &
        (rfm[column] <= upper_limit)
    ]


print("\nRFM Shape after removing outliers:")
print(rfm.shape)


# ============================================================
# 17. LOG TRANSFORMATION
# ============================================================

rfm_log = np.log1p(
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
)


# ============================================================
# 18. STANDARDIZATION
# ============================================================

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm_log
)


print("\nScaled RFM Shape:")
print(rfm_scaled.shape)


# ============================================================
# 19. CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(8, 6))

sns.heatmap(
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("RFM Correlation Heatmap")

plt.show()


# ============================================================
# 20. K-MEANS - ELBOW METHOD
# ============================================================

inertia = []

K_values = range(2, 11)

for k in K_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(rfm_scaled)

    inertia.append(
        model.inertia_
    )


plt.figure(figsize=(8, 5))

plt.plot(
    K_values,
    inertia,
    marker="o"
)

plt.title("K-Means Elbow Method")

plt.xlabel("Number of Clusters")

plt.ylabel("Inertia")

plt.xticks(K_values)

plt.show()


# ============================================================
# 21. K-MEANS - SILHOUETTE METHOD
# ============================================================

silhouette_values = []

for k in K_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        rfm_scaled
    )

    score = silhouette_score(
        rfm_scaled,
        labels
    )

    silhouette_values.append(score)


plt.figure(figsize=(8, 5))

plt.plot(
    K_values,
    silhouette_values,
    marker="o"
)

plt.title(
    "K-Means Silhouette Score"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(K_values)

plt.show()


# ============================================================
# 22. FINAL K-MEANS MODEL
# ============================================================

# You can change this value based on the elbow/silhouette graph.

best_k = 4

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

rfm["KMeans_Cluster"] = (
    kmeans.fit_predict(
        rfm_scaled
    )
)


# ============================================================
# 23. K-MEANS SCORE
# ============================================================

kmeans_score = silhouette_score(
    rfm_scaled,
    rfm["KMeans_Cluster"]
)

print("\nK-Means Silhouette Score:")
print(kmeans_score)


# ============================================================
# 24. K-MEANS CLUSTER SUMMARY
# ============================================================

print("\nK-Means Cluster Summary:")

kmeans_summary = (
    rfm
    .groupby("KMeans_Cluster")
    [
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .mean()
)

print(kmeans_summary)


# ============================================================
# 25. K-MEANS GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="KMeans_Cluster",
    palette="viridis"
)

plt.title(
    "K-Means Customer Segmentation"
)

plt.xlabel(
    "Frequency"
)

plt.ylabel(
    "Monetary"
)

plt.legend(
    title="Cluster"
)

plt.show()


# ============================================================
# 26. HIERARCHICAL CLUSTERING - DENDROGRAM
# ============================================================

# To avoid a huge dendrogram, use a sample.

sample_size = min(
    1000,
    len(rfm_scaled)
)

sample_indices = np.random.RandomState(
    42
).choice(
    len(rfm_scaled),
    size=sample_size,
    replace=False
)

sample_data = rfm_scaled[
    sample_indices
]


linkage_matrix = linkage(
    sample_data,
    method="ward"
)


plt.figure(figsize=(12, 6))

dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=30
)

plt.title(
    "Hierarchical Clustering Dendrogram"
)

plt.xlabel(
    "Customers"
)

plt.ylabel(
    "Distance"
)

plt.show()


# ============================================================
# 27. HIERARCHICAL CLUSTERING
# ============================================================

hierarchical = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward"
)

rfm["Hierarchical_Cluster"] = (
    hierarchical.fit_predict(
        rfm_scaled
    )
)


# ============================================================
# 28. HIERARCHICAL SCORE
# ============================================================

hierarchical_score = silhouette_score(
    rfm_scaled,
    rfm["Hierarchical_Cluster"]
)

print(
    "\nHierarchical Silhouette Score:"
)

print(
    hierarchical_score
)


# ============================================================
# 29. HIERARCHICAL SUMMARY
# ============================================================

print(
    "\nHierarchical Cluster Summary:"
)

hierarchical_summary = (
    rfm
    .groupby("Hierarchical_Cluster")
    [
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .mean()
)

print(
    hierarchical_summary
)


# ============================================================
# 30. HIERARCHICAL GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Hierarchical_Cluster",
    palette="viridis"
)

plt.title(
    "Hierarchical Customer Segmentation"
)

plt.xlabel(
    "Frequency"
)

plt.ylabel(
    "Monetary"
)

plt.legend(
    title="Cluster"
)

plt.show()


# ============================================================
# 31. DBSCAN
# ============================================================

dbscan = DBSCAN(
    eps=0.5,
    min_samples=5
)

dbscan_labels = dbscan.fit_predict(
    rfm_scaled
)

rfm["DBSCAN_Cluster"] = (
    dbscan_labels
)


# ============================================================
# 32. DBSCAN CLUSTER COUNTS
# ============================================================

print(
    "\nDBSCAN Cluster Counts:"
)

print(
    rfm["DBSCAN_Cluster"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 33. DBSCAN SILHOUETTE SCORE
# ============================================================

# -1 represents noise

dbscan_mask = (
    dbscan_labels != -1
)

unique_dbscan = set(
    dbscan_labels[dbscan_mask]
)

if len(unique_dbscan) >= 2:

    dbscan_score = silhouette_score(
        rfm_scaled[dbscan_mask],
        dbscan_labels[dbscan_mask]
    )

    print(
        "\nDBSCAN Silhouette Score:"
    )

    print(
        dbscan_score
    )

else:

    dbscan_score = np.nan

    print(
        "\nDBSCAN did not create enough clusters for silhouette score."
    )


# ============================================================
# 34. DBSCAN GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="DBSCAN_Cluster",
    palette="viridis"
)

plt.title(
    "DBSCAN Customer Segmentation"
)

plt.xlabel(
    "Frequency"
)

plt.ylabel(
    "Monetary"
)

plt.legend(
    title="Cluster"
)

plt.show()


# ============================================================
# 35. GAUSSIAN MIXTURE MODEL
# ============================================================

gmm = GaussianMixture(
    n_components=4,
    random_state=42
)

gmm_labels = gmm.fit_predict(
    rfm_scaled
)

rfm["GMM_Cluster"] = (
    gmm_labels
)


# ============================================================
# 36. GMM SILHOUETTE SCORE
# ============================================================

gmm_score = silhouette_score(
    rfm_scaled,
    gmm_labels
)

print(
    "\nGMM Silhouette Score:"
)

print(
    gmm_score
)


# ============================================================
# 37. GMM SUMMARY
# ============================================================

print(
    "\nGMM Cluster Summary:"
)

gmm_summary = (
    rfm
    .groupby("GMM_Cluster")
    [
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .mean()
)

print(
    gmm_summary
)


# ============================================================
# 38. GMM GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="GMM_Cluster",
    palette="viridis"
)

plt.title(
    "Gaussian Mixture Model Customer Segmentation"
)

plt.xlabel(
    "Frequency"
)

plt.ylabel(
    "Monetary"
)

plt.legend(
    title="Cluster"
)

plt.show()


# ============================================================
# 39. MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Algorithm": [
        "K-Means",
        "Hierarchical",
        "DBSCAN",
        "GMM"
    ],

    "Silhouette Score": [
        kmeans_score,
        hierarchical_score,
        dbscan_score,
        gmm_score
    ]

})


print("\n")
print("=" * 60)
print("CLUSTERING MODEL COMPARISON")
print("=" * 60)

print(
    comparison
)


# ============================================================
# 40. MODEL COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

sns.barplot(
    data=comparison,
    x="Algorithm",
    y="Silhouette Score"
)

plt.title(
    "Clustering Algorithm Comparison"
)

plt.xlabel(
    "Algorithm"
)

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(
    rotation=20
)

plt.show()


# ============================================================
# 41. CUSTOMER CLUSTER SIZE
# ============================================================

plt.figure(figsize=(9, 6))

sns.countplot(
    data=rfm,
    x="KMeans_Cluster"
)

plt.title(
    "Number of Customers in Each K-Means Cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Customers"
)

plt.show()


# ============================================================
# 42. RFM CLUSTER HEATMAP
# ============================================================

cluster_rfm = (
    rfm
    .groupby("KMeans_Cluster")
    [
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .mean()
)


plt.figure(figsize=(9, 6))

sns.heatmap(
    cluster_rfm,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu"
)

plt.title(
    "K-Means Cluster RFM Heatmap"
)

plt.xlabel(
    "RFM Features"
)

plt.ylabel(
    "Cluster"
)

plt.show()


# ============================================================
# 43. SAVE FINAL DATASET
# ============================================================

rfm.to_csv(
    "customer_clustering_results.csv"
)

print(
    "\nFinal clustering results saved as:"
)

print(
    "customer_clustering_results.csv"
)
